# 009 - Análise de desvios de previsão

## Objetivo e limites

Repetir a comparação do 008 com todas as variáveis e sem `PC1B`, `HA1A`, `IN15` e `IN22`, mantendo `SEX`, os mesmos folds e **124 árvores em ambas as versões**. A sinalização exploratória é definida antes da análise: uma previsão out-of-fold (OOF) fica sinalizada para revisão se diferir do escalão real em pelo menos **3 classes**. O sinal indica apenas um desvio do modelo, não uma anomalia real confirmada.

A análise usa apenas o conjunto de treino. Os perfis individuais não serão publicados; descrevemos padrões agregados. Os nomes de variáveis socioeconómicas, de saúde e de hábitos são tratados como candidatos e não como significados confirmados, pois não encontrámos um dicionário local.

## Preparação fixa e divisão

Usamos `preparar_dados_fixos()` apenas no CSV rotulado, sem ler o teste. A divisão 80/20 é a mesma do 006–008 (`random_state=42`); só a parcela de treino entra nos folds. Os folds estratificados (3, `shuffle=True`, `random_state=42`) são materializados uma vez e reutilizados sem alteração nas duas versões. Toda a imputação e codificação aprendida é ajustada dentro de cada fold.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import xgboost
from sklearn.model_selection import StratifiedKFold

candidate_roots = [Path.cwd(), *Path.cwd().resolve().parents]
repo_root = next(
    (p for p in candidate_roots if (p / 'notebooks').is_dir() and (p / 'src' / 'preparar_dados.py').is_file()),
    None,
)
if repo_root is None:
    raise FileNotFoundError('Não foi possível localizar a raiz do projeto com notebooks/ e src/preparar_dados.py.')
sys.path.insert(0, str(repo_root))

from src.preparar_dados import make_model_pipeline, make_stratified_split, preparar_dados_fixos

train_path = repo_root / 'notebooks' / 'dataset_train.csv'
if not train_path.is_file():
    raise FileNotFoundError(f'CSV de treino não encontrado: {train_path}')

print(f'Python: {sys.version}')
print(f'scikit-learn: {sklearn.__version__}; XGBoost: {xgboost.__version__}')
print(f'Raiz do projeto: {repo_root}')
train_df = pd.read_csv(train_path)
prepared_train = preparar_dados_fixos(train_df)
X_all, y_all = prepared_train['X'], prepared_train['y']
X_train, X_val, y_train, y_val = make_stratified_split(
    X_all, y_all, test_size=0.2, random_state=42
)
assert len(X_train) == 8769 and len(X_val) == 2193
assert X_train.index.equals(y_train.index)
assert sorted(y_train.unique().tolist()) == list(range(16))
print(f'Treino da experiência: {X_train.shape}; validação reservada: {X_val.shape}')
print(f'Folds: 3 estratificados; 124 árvores fixas; classes: {sorted(y_train.unique().tolist())}')

FOLD_SPLITS = list(StratifiedKFold(n_splits=3, shuffle=True, random_state=42).split(X_train, y_train))
assert len(FOLD_SPLITS) == 3
CONDITIONAL_COLUMNS = ['PC1B', 'HA1A', 'IN15', 'IN22']
missing_conditional = [column for column in CONDITIONAL_COLUMNS if column not in X_train.columns]
if missing_conditional:
    raise ValueError(f'Colunas necessárias para ablação ausentes: {missing_conditional}')
X_without_conditional = X_train.drop(columns=CONDITIONAL_COLUMNS)
assert 'SEX' in X_without_conditional.columns

Python: 3.13.5 | packaged by Anaconda, Inc. | (main, Jun 12 2025, 11:23:37) [Clang 14.0.6 ]
scikit-learn: 1.6.1; XGBoost: 3.4.1
Raiz do projeto: /Users/eduardaferreira/daa-kaggle-2026-2027


Treino da experiência: (8769, 236); validação reservada: (2193, 236)
Folds: 3 estratificados; 124 árvores fixas; classes: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]


## Configuração comparável ao 008 e folds OOF

Fixamos a configuração selecionada no 008 (`max_depth=5`, `learning_rate=0.1`, `min_child_weight=5`, `subsample=0.8`, `colsample_bytree=0.8`) e **124 estimadores em cada ajuste**, sem novo early stopping. Em cada fold, uma pipeline nova ajusta o pré-processador e o modelo no treino do fold e prevê apenas a respetiva validação. OOOF significa que cada previsão foi produzida por um modelo que não ajustou nesse exemplo.

In [2]:
from sklearn.metrics import f1_score
from xgboost import XGBClassifier

FIXED_MODEL_PARAMS = {
    'objective': 'multi:softprob',
    'num_class': 16,
    'tree_method': 'hist',
    'n_estimators': 124,
    'max_depth': 5,
    'learning_rate': 0.1,
    'min_child_weight': 5,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'random_state': 42,
    'n_jobs': 1,
    'verbosity': 0,
}

def predict_oof_fixed_trees(X, y, fold_splits):
    predictions = np.full(len(X), -1, dtype=int)
    fold_rows = []
    for fold_number, (train_positions, validation_positions) in enumerate(fold_splits, start=1):
        fold_pipeline = make_model_pipeline(
            XGBClassifier(**FIXED_MODEL_PARAMS),
            normalize_numeric=False,
        )
        fold_pipeline.fit(X.iloc[train_positions], y.iloc[train_positions])
        fold_prediction = fold_pipeline.predict(X.iloc[validation_positions]).astype(int)
        predictions[validation_positions] = fold_prediction
        fold_score = f1_score(
            y.iloc[validation_positions],
            fold_prediction,
            labels=list(range(16)),
            average='macro',
            zero_division=0,
        )
        fold_rows.append({'fold': fold_number, 'macro_f1': fold_score, 'n_estimators': 124})
    assert (predictions >= 0).all(), 'Nem todos os exemplos receberam uma previsão OOF.'
    return fold_rows, predictions

all_fold_scores, oof_all = predict_oof_fixed_trees(X_train, y_train, FOLD_SPLITS)
reduced_fold_scores, oof_reduced = predict_oof_fixed_trees(
    X_without_conditional, y_train, FOLD_SPLITS
)
assert len(oof_all) == len(oof_reduced) == len(y_train)

oof_score_table = pd.DataFrame([
    {
        'versao': 'Todas as variáveis',
        'fold_macro_f1': [row['macro_f1'] for row in all_fold_scores],
        'macro_f1_media_folds': np.mean([row['macro_f1'] for row in all_fold_scores]),
        'macro_f1_desvio_padrao': np.std([row['macro_f1'] for row in all_fold_scores], ddof=0),
        'macro_f1_oof_concatenado': f1_score(y_train, oof_all, labels=list(range(16)), average='macro', zero_division=0),
    },
    {
        'versao': 'Sem PC1B, HA1A, IN15, IN22 (SEX mantida)',
        'fold_macro_f1': [row['macro_f1'] for row in reduced_fold_scores],
        'macro_f1_media_folds': np.mean([row['macro_f1'] for row in reduced_fold_scores]),
        'macro_f1_desvio_padrao': np.std([row['macro_f1'] for row in reduced_fold_scores], ddof=0),
        'macro_f1_oof_concatenado': f1_score(y_train, oof_reduced, labels=list(range(16)), average='macro', zero_division=0),
    },
])
display(oof_score_table)

,versao,fold_macro_f1,macro_f1_media_folds,macro_f1_desvio_padrao,macro_f1_oof_concatenado
0,Todas as variáveis,"[0.45158618216928526, 0.4381928487679399, 0.45...",0.447710,0.006769,0.448007
1,"Sem PC1B, HA1A, IN15, IN22 (SEX mantida)","[0.43653731646684435, 0.4312834015426835, 0.44...",0.439123,0.007677,0.439373


## Sinalização pré-definida e comparação de casos

Definição fixa: desvio `= previsão OOF − classe real`; `desvio >= 3` significa previsão pelo menos três escalões acima da idade declarada; `desvio <= -3` significa previsão pelo menos três escalões abaixo. Os casos sinalizados são a união dos dois sentidos. Comparamos interseção, casos exclusivos, total de casos alterados e respetiva direção. A tabela de casos usa apenas o índice original da linha, não `PID`/`HHID`.

In [3]:
FLAG_THRESHOLD = 3
y_array = y_train.to_numpy(dtype=int)
delta_all = oof_all - y_array
delta_reduced = oof_reduced - y_array

above_all = delta_all >= FLAG_THRESHOLD
below_all = delta_all <= -FLAG_THRESHOLD
above_reduced = delta_reduced >= FLAG_THRESHOLD
below_reduced = delta_reduced <= -FLAG_THRESHOLD
flagged_all = above_all | below_all
flagged_reduced = above_reduced | below_reduced
both_flagged = flagged_all & flagged_reduced
all_only = flagged_all & ~flagged_reduced
reduced_only = ~flagged_all & flagged_reduced
neither_flagged = ~flagged_all & ~flagged_reduced

def describe_flags(label, flagged, above, below, delta):
    return {
        'versao': label,
        'sinalizados_total': int(flagged.sum()),
        'previstos_acima_3_ou_mais': int(above.sum()),
        'previstos_abaixo_3_ou_mais': int(below.sum()),
        'desvio_absoluto_medio_sinalizados': float(np.abs(delta[flagged]).mean()) if flagged.any() else np.nan,
    }

display(pd.DataFrame([
    describe_flags('Todas as variáveis', flagged_all, above_all, below_all, delta_all),
    describe_flags('Sem as quatro perguntas', flagged_reduced, above_reduced, below_reduced, delta_reduced),
]))
distance_summary = pd.DataFrame([
    {
        'versao': version,
        'desvio_3_escaloes': int((np.abs(delta) == 3).sum()),
        'desvio_4_escaloes': int((np.abs(delta) == 4).sum()),
        'desvio_5_ou_mais': int((np.abs(delta) >= 5).sum()),
    }
    for version, delta in [('Todas as variáveis', delta_all), ('Sem as quatro perguntas', delta_reduced)]
])
display(distance_summary)

case_comparison = pd.DataFrame([
    {'comparacao': 'Sinalizado em ambas (interseção)', 'casos': int(both_flagged.sum())},
    {'comparacao': 'Só com todas as variáveis', 'casos': int(all_only.sum())},
    {'comparacao': 'Só sem as quatro perguntas', 'casos': int(reduced_only.sum())},
    {'comparacao': 'Não sinalizado em nenhuma', 'casos': int(neither_flagged.sum())},
    {'comparacao': 'Total com sinalização diferente', 'casos': int((all_only | reduced_only).sum())},
    {'comparacao': 'Sinalizados com direção diferente', 'casos': int((flagged_all & flagged_reduced & (np.sign(delta_all) != np.sign(delta_reduced))).sum())},
    {'comparacao': 'Classe prevista diferente, qualquer distância', 'casos': int((oof_all != oof_reduced).sum())},
])
display(case_comparison)

case_table = pd.DataFrame({
    'indice_original': X_train.index.to_numpy(),
    'AGE_RANGE_real': y_array,
    'previsto_todas': oof_all,
    'desvio_todas': delta_all,
    'sinalizado_todas': flagged_all,
    'previsto_sem_condicionais': oof_reduced,
    'desvio_sem_condicionais': delta_reduced,
    'sinalizado_sem_condicionais': flagged_reduced,
})
changed_flag_cases = case_table.loc[
    case_table['sinalizado_todas'] != case_table['sinalizado_sem_condicionais']
].copy()
print('Casos cuja sinalização muda (primeiros 25, ordenados pelo maior desvio):')
display(changed_flag_cases.assign(
    maior_desvio=lambda frame: frame[['desvio_todas', 'desvio_sem_condicionais']].abs().max(axis=1)
).sort_values('maior_desvio', ascending=False).head(25))

,versao,sinalizados_total,previstos_acima_3_ou_mais,previstos_abaixo_3_ou_mais,desvio_absoluto_medio_sinalizados
0,Todas as variáveis,416,216,200,3.399038
1,Sem as quatro perguntas,423,212,211,3.397163


,versao,desvio_3_escaloes,desvio_4_escaloes,desvio_5_ou_mais
0,Todas as variáveis,289,99,28
1,Sem as quatro perguntas,295,98,30


,comparacao,casos
0,Sinalizado em ambas (interseção),307
1,Só com todas as variáveis,109
2,Só sem as quatro perguntas,116
3,Não sinalizado em nenhuma,8237
4,Total com sinalização diferente,225
5,Sinalizados com direção diferente,0
6,"Classe prevista diferente, qualquer distância",1639


Casos cuja sinalização muda (primeiros 25, ordenados pelo maior desvio):


,indice_original,AGE_RANGE_real,previsto_todas,desvio_todas,sinalizado_todas,previsto_sem_condicionais,desvio_sem_condicionais,sinalizado_sem_condicionais,maior_desvio
7451,4624,1,2,1,False,7,6,True,6
7729,3624,9,15,6,True,9,0,False,6
6389,9000,2,8,6,True,4,2,False,6
4265,5573,8,6,-2,False,3,-5,True,5
1541,2376,7,7,0,False,2,-5,True,5
4072,2534,2,7,5,True,3,1,False,5
7408,3194,2,7,5,True,4,2,False,5
5383,10707,3,4,1,False,8,5,True,5
7557,9124,15,14,-1,False,10,-5,True,5
5456,3115,15,10,-5,True,15,0,False,5


## Caracterização agregada dos desvios maiores

Para descrever os casos sinalizados, comparamos perfis agregados nos dois sentidos (`>=3` escalões acima e `<=-3` abaixo) com o restante treino OOF. As variáveis abaixo são candidatas socioeconómicas, de saúde e de hábitos pelo código/nome; sem dicionário, não atribuímos significado substantivo nem interpretamos categorias desconhecidas. Exibimos as categorias mais frequentes em cada grupo e a respetiva proporção, nunca linhas individuais nem identificadores. As associações são descritivas e não explicam causalmente o erro.

In [4]:
profile_columns = {
    'socioeconomia_candidatas': ['DEG_URB', 'HATLEVEL', 'MAINSTAT', 'HHINCOME', 'HHNBPERS_COD', 'MARSTALEGAL'],
    'saude_candidatas': ['HS1', 'HS2', 'HS3', 'AC1A', 'AC1B', 'AC1C', 'BM1', 'BM2'],
    'habitos_candidatos': ['AL1', 'AL2', 'AL3', 'AL4', 'AL5', 'AL6', 'SK2', 'SK3', 'PE1', 'PE2', 'PE3'],
}
available_profile_columns = {
    group: [column for column in columns if column in train_df.columns]
    for group, columns in profile_columns.items()
}
profile_base = train_df.loc[X_train.index]
profile_rows = []
profile_groups = {
    'todas_acima_3_ou_mais': above_all,
    'todas_abaixo_3_ou_mais': below_all,
    'sem_condicionais_acima_3_ou_mais': above_reduced,
    'sem_condicionais_abaixo_3_ou_mais': below_reduced,
    'restante_treino_oof': ~(flagged_all | flagged_reduced),
}
for direction, mask in profile_groups.items():
    group_frame = profile_base.iloc[np.flatnonzero(mask)]
    for domain, columns in available_profile_columns.items():
        for column in columns:
            values = group_frame[column].astype('string').fillna('Ausente')
            frequencies = values.value_counts(normalize=True, dropna=False).head(3)
            for category, proportion in frequencies.items():
                profile_rows.append({
                    'direcao_desvio': direction,
                    'grupo_candidato': domain,
                    'variavel': column,
                    'categoria_observada': str(category),
                    'proporcao_no_grupo': float(proportion),
                    'n_grupo': int(len(group_frame)),
                })
profile_summary = pd.DataFrame(profile_rows)
selected_profile_columns = ['DEG_URB', 'HATLEVEL', 'MAINSTAT', 'HHINCOME', 'HS1', 'HS2', 'HS3', 'AL1', 'SK2', 'PE1']
selected_profile_summary = profile_summary.loc[
    profile_summary['variavel'].isin(selected_profile_columns)
].reset_index(drop=True)
with pd.option_context('display.max_rows', 200, 'display.max_columns', None):
    display(selected_profile_summary)
print('Nota: as designações dos grupos são exploratórias; confirmar com o dicionário do questionário antes de interpretação substantiva.')

,direcao_desvio,grupo_candidato,variavel,categoria_observada,proporcao_no_grupo,n_grupo
0,todas_acima_3_ou_mais,socioeconomia_candidatas,DEG_URB,Áreas mediamente povoadas,0.430556,216
1,todas_acima_3_ou_mais,socioeconomia_candidatas,DEG_URB,Áreas pouco povoadas,0.300926,216
2,todas_acima_3_ou_mais,socioeconomia_candidatas,DEG_URB,Áreas densamente povoadas,0.268519,216
3,todas_acima_3_ou_mais,socioeconomia_candidatas,HATLEVEL,Básico 1º e 2º ciclo,0.268519,216
4,todas_acima_3_ou_mais,socioeconomia_candidatas,HATLEVEL,Básico 3º ciclo,0.212963,216
5,todas_acima_3_ou_mais,socioeconomia_candidatas,HATLEVEL,Secundário,0.212963,216
6,todas_acima_3_ou_mais,socioeconomia_candidatas,MAINSTAT,Tem um emprego ou trabalho (inc. trabalhador f...,0.500000,216
7,todas_acima_3_ou_mais,socioeconomia_candidatas,MAINSTAT,"Reformado, com reforma antecipada ou cessou a ...",0.226852,216
8,todas_acima_3_ou_mais,socioeconomia_candidatas,MAINSTAT,Desempregado,0.138889,216
9,todas_acima_3_ou_mais,socioeconomia_candidatas,HHINCOME,1º quintil,0.263889,216


Nota: as designações dos grupos são exploratórias; confirmar com o dicionário do questionário antes de interpretação substantiva.


## Conclusão

Com 124 árvores fixas e os mesmos folds, a Macro F1 OOF média foi **0,4477** (DP **0,0068**) com todas as variáveis e **0,4391** (DP **0,0077**) sem as quatro perguntas; a diferença é **+0,0086** a favor do conjunto completo. O limiar `|desvio| >= 3` sinalizou **416** casos com todas as variáveis (216 acima, 200 abaixo) e **423** sem as perguntas (212 acima, 211 abaixo). **307** casos foram sinalizados por ambas, **109** apenas pelo modelo completo e **116** apenas pelo reduzido; em **1.639** casos a classe prevista mudou, ainda que muitas mudanças não cruzassem o limiar. Não houve inversões de direção entre casos sinalizados por ambos os modelos. A maioria dos desvios sinalizados foi exatamente de três escalões (289 e 295, respetivamente); desvios de cinco ou mais escalões foram menos frequentes (28 e 30).

Como exemplo descritivo, entre os casos em que o modelo completo prevê pelo menos três escalões acima, as categorias mais frequentes incluem `DEG_URB` «Áreas mediamente povoadas» (**43,1%**) e `HATLEVEL` «Básico 1.º e 2.º ciclo» (**26,9%**). As tabelas agregadas também mostram candidatos de saúde e hábitos; sem dicionário, estes códigos e proporções não permitem inferir causas ou perfis anómalos confirmados.

O modelo completo é um candidato razoável para futura submissão por ter melhor Macro F1 OOF e manter uma contagem de casos distantes semelhante; a remoção das perguntas troca muitos casos individuais sem melhorar o score. Esta análise OOF não substitui a validação fixa nem a confirmação humana dos casos. **Não foi criado CSV.**